# AI Jobbmatchning V6

Notebook-version av programmet. Koden är indelad i logiska celler för att göra projektet enklare att läsa, testa och förklara.

## 1. Importer

In [ ]:
import requests              # Importerar requests för HTTP-anrop till API:t.
import json                  # Importerar json för att läsa och spara JSON-data.
from datetime import datetime  # Importerar datetime för datum och tid.


## 2. Hjälpmetoder

In [ ]:
def normalisera(text):                  # Skapar en funktion som normaliserar text.
    return text.lower().strip()         # Gör texten till små bokstäver och tar bort ytterkanter av mellanslag.


def visa_kompetenser(kompetenser):     # Skapar en funktion som visar kompetenser som text.
    return ", ".join(kompetenser) if kompetenser else "Inga"  # Slår ihop kompetenserna eller visar Inga om listan är tom.


## 3. Basklass och profil

In [ ]:
class MatchningsData:                                      # Skapar basklassen för gemensamma uppgifter.
    def __init__(self):                                    # Skapar objektets startvärden.
        self.plats = []                                   # Skapar en tom lista för platser.
        self.kompetenser = []                             # Skapar en tom lista för kompetenser.


class Profil(MatchningsData):                             # Skapar Profil som ärver från MatchningsData.
    def __init__(self):                                   # Skapar objektets startvärden.
        super().__init__()                                # Kör basklassens konstruktor.
        self.jobb = []                                    # Skapar en tom lista för önskade jobb.

    def input_jobb(self):                                 # Skapar en funktion för att mata in önskade jobb.
        self._input("Vad hade du velat jobba som? ", "jobb")  # Hämtar användarens önskade jobb.

    def input_plats(self):                                # Skapar en funktion för att mata in önskade platser.
        self._input("Vart hade du velat jobba? ", "plats")  # Hämtar användarens önskade platser.

    def input_kompetenser(self):                          # Skapar en funktion för att mata in kompetenser.
        self._input("Vad har du för kompetenser? ", "kompetenser")  # Hämtar användarens kompetenser.

    def _input(self, text, attribut):                     # Skapar en gemensam funktion för användarinmatning.
        print("\n" + "=" * 40)                            # Skriver ut en avgränsare.

        try:                                              # Försöker köra kod som kan ge ett fel.
            värden = [                                    # Skapar en lista med normaliserade användarsvar.
                normalisera(x)                            # Normaliserar varje inmatat värde.
                for x in input(text).split(",")           # Läser in text och delar den vid kommatecken.
                if x.strip()                              # Tar bara med värden som inte är tomma.
            ]                                             # Avslutar listan med inmatade värden.
            getattr(self, attribut).extend(värden)        # Lägger de nya värdena i rätt lista.

        except (KeyboardInterrupt, EOFError):             # Fångar avbruten eller avslutad användarinmatning.
            print("\nInmatningen avbröts.")               # Informerar användaren att inmatningen avbröts.


## 4. Jobbannons och matchare

In [ ]:
class Jobbannons(MatchningsData):                         # Skapar Jobbannons som ärver från MatchningsData.
    def __init__(self, jobb, plats, kompetenser, länk):  # Tar emot information om en jobbannons.
        super().__init__()                                # Kör basklassens konstruktor.
        self.jobb = jobb                                  # Sparar jobbets titel.
        self.plats = plats                                # Sparar jobbets plats.
        self.kompetenser = kompetenser                    # Sparar matchande kompetenser.
        self.länk = länk                                  # Sparar länken till jobbannonsen.


class Matchare:                                          # Skapar klassen som ansvarar för matchningen.
    def __init__(self, profil, arbetsmarknad):            # Tar emot profilen och jobbannonserna.
        self.profil = profil                              # Sparar användarens profil.
        self.arbetsmarknad = arbetsmarknad                # Sparar listan med jobbannonser.

    def match(self):                                      # Skapar funktionen som räknar ut matchningar.
        matchningar = []                                  # Skapar en tom lista för matchningsresultat.

        for jobb in self.arbetsmarknad:                   # Går igenom alla jobbannonser.
            jobb_matchar = False                          # Börjar med att anta att jobbet inte matchar.

            for önskat_jobb in self.profil.jobb:          # Går igenom användarens önskade jobb.
                if önskat_jobb in jobb.jobb:              # Kontrollerar om jobbets titel matchar.
                    jobb_matchar = True                   # Markerar att jobbtiteln matchar.
                    break                                 # Avslutar loopen när en match hittats.

            if not jobb_matchar:                          # Kontrollerar om jobbet inte matchar.
                continue                                  # Hoppar vidare till nästa jobb.

            poäng = 0                                     # Börjar matchningspoängen på noll.

            for plats in self.profil.plats:               # Går igenom användarens önskade platser.
                if plats in jobb.plats:                   # Kontrollerar om platsen matchar.
                    poäng += 2                             # Ger två poäng för rätt plats.
                    break                                 # Avslutar loopen när en match hittats.

            for kompetens in self.profil.kompetenser:     # Går igenom användarens kompetenser.
                if kompetens in jobb.kompetenser:         # Kontrollerar om kompetensen matchar.
                    poäng += 1                             # Ger en poäng för varje matchande kompetens.

            if poäng == 0:                                 # Kontrollerar om bara jobbtiteln matchade.
                poäng = 1                                  # Ger en grundpoäng för titelmatchningen.

            matchningar.append((jobb, poäng))             # Sparar jobbet och dess poäng.

        matchningar.sort(                                 # Sorterar matchningarna efter poäng.
            key=lambda x: x[1],                           # Anger att poängen ska användas vid sorteringen.
            reverse=True                                  # Sorterar från högsta till lägsta poäng.
        )                                                 # Kodrad som används för programmets funktion.

        return matchningar                                # Returnerar de färdiga matchningarna.


## 5. Hämta jobb från JobTech API

In [ ]:
def hämta_jobb_från_api(profil):                         # Skapar funktionen som hämtar jobb från API:t.
    if not profil.jobb:                                  # Kontrollerar om användaren har valt något jobb.
        print("Du måste ställa in din profil först.")    # Informerar användaren att profilen måste fyllas i.
        return []                                        # Returnerar en tom lista när ingen profil finns.

    url = "https://jobsearch.api.jobtechdev.se/search"   # Sparar adressen till JobTechs API.
    jobbannonser = []                                    # Skapar en lista för jobbannonser.
    alla_data = []                                       # Skapar en lista för rådata från API:t.

    try:                                                 # Försöker köra kod som kan ge ett fel.
        for sökord in profil.jobb:                       # Söker efter varje önskat jobb.
            print(f"\nSöker efter: {sökord}")            # Visar vilket sökord som används.

            response = requests.get(                    # Skickar ett GET-anrop till API:t.
                url,                                     # Anger API-adressen som ska anropas.
                params={"q": sökord, "limit": 10},       # Skickar sökord och begränsar resultatet till tio annonser.
                timeout=10                               # Avbryter anropet om API:t inte svarar inom tio sekunder.
            )                                            # Kodrad som används för programmets funktion.

            print("Statuskod:", response.status_code)   # Visar HTTP-statuskoden från API:t.

            if response.status_code != 200:              # Kontrollerar om API-anropet lyckades.
                print("Kunde inte hämta jobbannonser.")  # Informerar om att annonserna inte kunde hämtas.
                continue                                 # Hoppar vidare till nästa jobb.

            data = response.json()                       # Omvandlar API-svaret från JSON till Python-data.
            alla_data.append(data)                       # Sparar API-svaret i listan med rådata.

            träffar = data.get("hits", [])                # Hämtar jobbträffarna från API-svaret.
            print("Antal träffar:", len(träffar))        # Visar antalet hittade jobb.

            for jobb in träffar:                         # Går igenom varje hittad jobbannons.
                titel = normalisera(                     # Hämtar och normaliserar jobbets titel.
                    jobb.get("headline") or ""           # Hämtar rubriken eller använder tom text om den saknas.
                )                                        # Kodrad som används för programmets funktion.

                plats = normalisera(                     # Hämtar och normaliserar jobbets plats.
                    (jobb.get("workplace_address") or {}) # Hämtar arbetsplatsens adress eller en tom dictionary.
                    .get("municipality") or ""            # Hämtar kommunen eller använder tom text.
                )                                        # Kodrad som används för programmets funktion.

                länk = jobb.get("webpage_url") or ""     # Hämtar länken till jobbannonsen.

                beskrivning = normalisera(               # Hämtar och normaliserar jobbets beskrivning.
                    jobb.get("description", {})           # Hämtar beskrivningsobjektet.
                    .get("text") or ""                    # Hämtar beskrivningens text eller tom text.
                )                                        # Kodrad som används för programmets funktion.

                matchade_kompetenser = []                # Skapar en lista för kompetenser som matchar.

                for kompetens in profil.kompetenser:     # Går igenom användarens kompetenser.
                    if kompetens in beskrivning:         # Kontrollerar om kompetensen finns i beskrivningen.
                        matchade_kompetenser.append(kompetens)  # Lägger till den matchande kompetensen.

                jobbannonser.append(                     # Lägger till en ny jobbannons i listan.
                    Jobbannons(                           # Skapar ett Jobbannons-objekt.
                        titel,                            # Skickar med jobbets titel.
                        plats,                            # Skickar med jobbets plats.
                        matchade_kompetenser,             # Skickar med matchande kompetenser.
                        länk                               # Skickar med länken till annonsen.
                    )                                    # Kodrad som används för programmets funktion.
                )                                        # Kodrad som används för programmets funktion.

        try:                                             # Försöker köra kod som kan ge ett fel.
            with open(                                   # Öppnar en fil.
                "jobbdata.json",                         # Anger filnamnet för sparad API-data.
                "w",                                     # Öppnar historikfilen i skrivläge.
                encoding="utf-8"                         # Använder UTF-8 för svenska tecken.
            ) as f:                                      # Kodrad som används för programmets funktion.
                json.dump(                               # Skriver Python-data till JSON-format.
                    alla_data,                           # Anger datan som ska sparas.
                    f,                                   # Anger filen som datan ska skrivas till.
                    ensure_ascii=False,                  # Behåller svenska tecken i JSON-filen.
                    indent=4                              # Formaterar JSON-filen med indrag.
                )                                        # Kodrad som används för programmets funktion.

        except OSError as fel:                            # Fångar fel vid filhantering.
            print("\nKunde inte spara jobbdata.")         # Informerar om att jobbdata inte kunde sparas.
            print(fel)                                   # Visar felmeddelandet.

        return jobbannonser                               # Returnerar de hämtade jobbannonserna.

    except requests.exceptions.RequestException as fel:  # Fångar nätverksfel från requests.
        print("\nEtt fel uppstod när API:t kontaktades.") # Informerar om API-felet.
        print(fel)                                       # Visar felmeddelandet.
        return []                                        # Returnerar en tom lista.

    except json.JSONDecodeError:                         # Fångar felaktig JSON-data.
        print("\nKunde inte läsa svaret från API:t.")    # Informerar om att API-svaret inte kunde läsas.
        return []                                        # Returnerar en tom lista.


## 6. Spara och visa historik

In [ ]:
def spara_historik(profil, matchningar):                 # Skapar funktionen som sparar historik.
    try:                                                 # Försöker köra kod som kan ge ett fel.
        with open(                                       # Öppnar en fil.
            "historik.json",                             # Anger historikfilen som ska läsas.
            "r",                                         # Öppnar filen i läsläge.
            encoding="utf-8"                             # Använder UTF-8 för svenska tecken.
        ) as f:                                          # Sparar den öppnade filen i variabeln f.
            historik = json.load(f)                      # Läser historiken från JSON till Python-data.

    except (FileNotFoundError, json.JSONDecodeError):    # Hanterar saknad eller trasig historikfil.
        historik = []                                    # Startar med en tom historik.

    ny_sökning = {                                       # Skapar en ny post för den aktuella sökningen.
        "datum": datetime.now().strftime("%Y-%m-%d %H:%M"),  # Sparar aktuellt datum och klockslag.
        "profil": {                                      # Skapar delen som innehåller användarens profil.
            "jobb": profil.jobb,                         # Sparar användarens önskade jobb.
            "plats": profil.plats,                       # Sparar användarens önskade platser.
            "kompetenser": profil.kompetenser            # Sparar användarens kompetenser.
        },                                               # Avslutar profilen.
        "matchningar": []                                # Skapar en tom lista för matchningarna.
    }                                                    # Avslutar den nya sökningen.

    for jobb, poäng in matchningar:                      # Går igenom jobb och deras matchningspoäng.
        ny_sökning["matchningar"].append({               # Lägger till en matchning i historiken.
            "jobb": jobb.jobb,                           # Sparar jobbets titel i historiken.
            "plats": jobb.plats,                         # Sparar jobbets plats i historiken.
            "poäng": poäng,                              # Sparar matchningspoängen.
            "kompetenser": jobb.kompetenser,             # Sparar matchande kompetenser.
            "länk": jobb.länk                             # Sparar länken till jobbet.
        })                                               # Avslutar matchningen.

    historik.append(ny_sökning)                          # Lägger den nya sökningen sist i historiken.

    try:                                                 # Försöker köra kod som kan ge ett fel.
        with open(                                       # Öppnar en fil.
            "historik.json",                             # Anger historikfilen som ska skrivas till.
            "w",                                         # Öppnar historikfilen i skrivläge.
            encoding="utf-8"                             # Använder UTF-8 för svenska tecken.
        ) as f:                                          # Sparar den öppnade filen i variabeln f.
            json.dump(                                   # Skriver Python-data till JSON-format.
                historik,                                # Anger historiken som ska sparas.
                f,                                        # Anger filen som datan ska skrivas till.
                ensure_ascii=False,                      # Behåller svenska tecken i JSON-filen.
                indent=4                                  # Formaterar JSON-filen med indrag.
            )                                            # Avslutar skrivningen till JSON-filen.

    except OSError as fel:                               # Fångar fel vid filhantering.
        print("\nKunde inte spara historiken.")          # Informerar om fel vid sparning.
        print(fel)                                       # Visar felmeddelandet.
        return                                           # Avslutar funktionen.

    print("\nSökningen har sparats i historiken.")       # Bekräftar att sökningen sparats.


def visa_historik():                                     # Skapar funktionen som visar historiken.
    try:                                                 # Försöker köra kod som kan ge ett fel.
        with open(                                       # Öppnar en fil.
            "historik.json",                             # Anger historikfilen som ska läsas.
            "r",                                         # Öppnar filen i läsläge.
            encoding="utf-8"                             # Använder UTF-8 för svenska tecken.
        ) as f:                                          # Sparar den öppnade filen i variabeln f.
            historik = json.load(f)                      # Läser historiken från JSON till Python-data.

    except FileNotFoundError:                            # Hanterar om historikfilen saknas.
        print("\nDet finns ingen historik ännu.")        # Informerar om att historik saknas.
        return                                           # Avslutar funktionen.

    except json.JSONDecodeError:                         # Fångar felaktig JSON-data.
        print("\nHistorikfilen kunde inte läsas.")       # Informerar om att historiken inte kan läsas.
        return                                           # Avslutar funktionen.

    if not historik:                                     # Kontrollerar om historiken är tom.
        print("\nDet finns ingen historik ännu.")        # Informerar om att historik saknas.
        return                                           # Avslutar funktionen.

    print("\n" + "=" * 40)                               # Skriver ut en avgränsare.
    print("              HISTORIK")                      # Skriver ut rubriken HISTORIK.
    print("=" * 40)                                      # Skriver ut en avgränsare.

    for nummer, sökning in enumerate(historik, 1):       # Går igenom varje sökning och ger den ett nummer.
        print(f"\nSökning {nummer}")                     # Visar numret på sökningen.
        print(f"Datum: {sökning['datum']}")              # Visar datumet för sökningen.
        print(f"Jobb: {', '.join(sökning['profil']['jobb'])}")  # Visar användarens önskade jobb.
        print(f"Plats: {', '.join(sökning['profil']['plats'])}")  # Visar användarens önskade platser.
        print(                                           # Skriver ut användarens kompetenser.
            f"Kompetenser: "                             # Skriver ut rubriken Kompetenser.
            f"{', '.join(sökning['profil']['kompetenser'])}"  # Visar användarens kompetenser.
        )                                                # Avslutar utskriften.

        print("\nMatchningar:")                          # Skriver ut rubriken för matchningarna.

        for jobb in sökning["matchningar"]:              # Går igenom alla matchningar i sökningen.
            print(f"  - {jobb['jobb']}")                # Visar jobbets titel.
            print(f"    Plats: {jobb['plats']}")        # Visar jobbets plats.
            print(f"    Poäng: {jobb['poäng']}")        # Visar jobbets matchningspoäng.
            print(                                       # Skriver ut matchande kompetenser.
                "    Matchande kompetenser: "            # Skriver ut rubriken för kompetenserna.
                + visa_kompetenser(jobb["kompetenser"]) # Visar de matchande kompetenserna.
            )                                            # Avslutar utskriften.
            print(f"    Länk: {jobb['länk']}")           # Visar länken till jobbannonsen.

        print("-" * 40)                                  # Skriver en avgränsare mellan sökningarna.


## 7. Visa matchningar

In [ ]:
def visa_matchningar(profil):                          # Skapar funktionen som hämtar och visar matchningar.
    print("\nHämtar aktuella jobbannonser...")          # Informerar att jobbannonser hämtas.

    jobb_fran_api = hämta_jobb_från_api(profil)         # Hämtar jobbannonser från API:t.

    if not jobb_fran_api:                               # Kontrollerar om några jobbannonser hämtades.
        print("Inga jobbannonser kunde hämtas.")        # Informerar om att inga annonser hämtades.
        return                                          # Avslutar funktionen.

    matchningar = Matchare(                             # Skapar en Matchare för profilen och annonserna.
        profil,                                         # Skickar in användarens profil.
        jobb_fran_api                                   # Skickar in jobbannonserna.
    ).match()                                           # Kör matchningsfunktionen.

    if not matchningar:                                 # Kontrollerar om några jobb matchade profilen.
        print("\nInga jobb matchade din profil.")       # Informerar om att ingen annons matchade.
        return                                          # Avslutar funktionen.

    spara_historik(profil, matchningar)                 # Sparar matchningarna i historiken.

    print("\n" + "=" * 40)                              # Skriver ut en avgränsare.
    print("           DINA MATCHNINGAR")                # Skriver ut rubriken för matchningsresultatet.
    print("=" * 40)                                     # Skriver ut en avgränsare.

    for jobb, poäng in matchningar:                     # Går igenom jobb och deras matchningspoäng.
        print(f"\n- {jobb.jobb}")                       # Visar jobbets titel.
        print(f"  Plats: {jobb.plats}")                # Visar jobbets plats.
        print(f"  Poäng: {poäng}")                      # Visar matchningspoängen.
        print(                                          # Skriver ut matchande kompetenser.
            "  Matchande kompetenser: "                 # Skriver ut rubriken för kompetenserna.
            + visa_kompetenser(jobb.kompetenser)        # Visar de matchande kompetenserna.
        )                                               # Avslutar utskriften.
        print(f"  Länk: {jobb.länk}")                   # Visar länken till jobbet.


## 8. Användargränssnitt

In [ ]:
profil = Profil()                                      # Skapar programmets profilobjekt.


def user_ui():                                         # Skapar programmets huvudmeny.
    while True:                                        # Kör menyn tills användaren avslutar.
        print("\n")                                    # Skapar en tom rad för bättre läsbarhet.
        print("=" * 40)                                # Skriver ut en avgränsare.
        print("       VÄLKOMMEN TILL AI-JOBBANALYS")  # Visar programmets titel.
        print("=" * 40)                                # Skriver ut en avgränsare.
        print("1. Ställ in profil")                    # Visar menyvalet för profil.
        print("2. Visa matchningar")                   # Visar menyvalet för matchningar.
        print("3. Historik")                           # Visar menyvalet för historik.
        print("4. Avsluta")                            # Visar menyvalet för att avsluta.
        print("=" * 40)                                # Skriver ut en avgränsare.

        try:                                           # Försöker köra kod som kan ge ett fel.
            svar = input("Välj ett alternativ 1-4: ")  # Läser användarens menyval.

        except (KeyboardInterrupt, EOFError):           # Fångar avbruten eller avslutad användarinmatning.
            print("\nProgrammet avslutas.")             # Informerar att programmet avslutas.
            break                                      # Avslutar menyn.

        if svar == "1":                                # Kontrollerar om användaren vill ställa in profilen.
            profil.input_jobb()                        # Läser in önskat jobb.
            profil.input_plats()                       # Läser in önskad plats.
            profil.input_kompetenser()                 # Läser in kompetenser.

            print("\nProfil ändrad!")                  # Bekräftar att profilen ändrats.
            print(f"\nJobb: {profil.jobb}")             # Visar valda jobb.
            print(f"Plats/Ort: {profil.plats}")        # Visar valda platser.
            print(f"Kompetenser: {profil.kompetenser}") # Visar valda kompetenser.

        elif svar == "2":                              # Kontrollerar om användaren vill se matchningar.
            visa_matchningar(profil)                   # Hämtar och visar matchande jobb.

        elif svar == "3":                              # Kontrollerar om användaren vill se historiken.
            visa_historik()                            # Visar tidigare sökningar.

        elif svar == "4":                              # Kontrollerar om användaren vill avsluta.
            print("\nTack för denna gång!")            # Visar ett avslutningsmeddelande.
            break                                      # Avslutar menyn.

        else:                                          # Hanterar ett menyval som inte är giltigt.
            print("\nFelaktigt val.")                  # Informerar om att menyvalet är fel.
            print("Välj en siffra mellan 1 och 4.")    # Ber användaren välja ett giltigt alternativ.


## 9. Starta programmet

In [ ]:

user_ui() # Startar programmets användargränssnitt.

## 10. Utvecklingsplan

1. Profil: Jobb, Plats, Kompetenser  
2. API: Hämta aktuella jobbannonser  
3. Matchning: Matcha jobb, plats, kompetenser  
4. Resultat: Visa jobb och matchningspoäng  
5. Förbättringar: Validering, Historik, Bättre UI  
6. Testning: Testa och färdigställ programmet